# Feature Characterization

Here I group the features by how similar they quantify image textures.

Below I created functions that would produce the GLCM that maximize/minimize each feature function. From here we can group the the features by the resulting GLCMs. Some features are maximized by:
- Distrubutions concentrated at the bottom corner ($N-1$, $N-1$ entry)
- Concentration at the main cross-diagonal
- Concentration at the diagonal (or off diagonal)
- Concentration at a single row
- Uniform distribution


In [12]:
import numpy as np
from scipy.optimize import minimize

In [13]:
def maximize_haralick(feature_func, n_levels):
    def objective(flat_p):
        P = flat_p.reshape((n_levels, n_levels))
        return -feature_func(P)

    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bnds = [(0, 1) for _ in range(n_levels**2)]
    init_guess = np.ones(n_levels**2) / (n_levels**2)
    
    res = minimize(objective, init_guess, method='SLSQP', bounds=bnds, constraints=cons)
    return res.x.reshape((n_levels, n_levels)), -res.fun


def minimize_haralick(feature_func, n_levels):
    def objective(flat_p):
        P = flat_p.reshape((n_levels, n_levels))
        return feature_func(P)

    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bnds = [(0, 1) for _ in range(n_levels**2)]
    init_guess = np.ones(n_levels**2) / (n_levels**2)
    
    res = minimize(objective, init_guess, method='SLSQP', bounds=bnds, constraints=cons)
    return res.x.reshape((n_levels, n_levels)), -res.fun

In [14]:
def get_stats(P):
    N = P.shape[0]
    # Ensure P is normalized (crucial for optimizer stability)
    P = P / np.sum(P)
    
    # Grids (1-based for formula consistency)
    i, j = np.indices((N, N))
    i, j = i + 1, j + 1
    
    # Marginal probabilities
    p_x = np.sum(P, axis=1)
    p_y = np.sum(P, axis=0)
    
    # Means
    mu_x = np.sum(np.arange(1, N + 1) * p_x)
    mu_y = np.sum(np.arange(1, N + 1) * p_y)
    mu = (mu_x + mu_y) / 2
    
    # Standard Deviations
    sigma_x = np.sqrt(np.sum(((np.arange(1, N + 1) - mu_x)**2) * p_x))
    sigma_y = np.sqrt(np.sum(((np.arange(1, N + 1) - mu_y)**2) * p_y))
    
    # p_x+y and p_x-y distributions
    p_sum = np.zeros(2 * N + 1)
    p_diff = np.zeros(N)
    for row in range(N):
        for col in range(N):
            p_sum[(row+1) + (col+1)] += P[row, col]
            p_diff[abs(row - col)] += P[row, col]
            
    return {
        'P': P, 'N': N, 'i': i, 'j': j, 
        'mu_x': mu_x, 'mu_y': mu_y, 'mu': mu,
        'sigma_x': sigma_x, 'sigma_y': sigma_y,
        'p_x': p_x, 'p_y': p_y, 'p_sum': p_sum, 'p_diff': p_diff
    }


def feat_autocorr(P):
    s = get_stats(P)
    return np.sum(s['i'] * s['j'] * s['P'])

def feat_contrast(P):
    s = get_stats(P)
    return np.sum(s['p_diff'] * (np.arange(s['N'])**2))

def feat_cluster_prominence(P):
    s = get_stats(P)
    return np.sum(P * (s['i'] + s['j'] - 2 * s['mu'])**4)

def feat_cluster_shade(P):
    s = get_stats(P)
    return np.sum(P * (s['i'] + s['j'] - 2 * s['mu'])**3)

def feat_cluster_tendency(P):
    s = get_stats(P)
    return np.sum(P * (s['i'] + s['j'] - 2 * s['mu'])**2)

def feat_entropy(P):
    # Small epsilon to avoid log(0)
    return -np.sum(P * np.log(P + 1e-15))

def feat_asm(P):
    return np.sum(P**2)

def feat_correlation(P):
    s = get_stats(P)
    if s['sigma_x'] * s['sigma_y'] == 0: return 0
    term = np.sum(s['i'] * s['j'] * P)
    return (term - (s['mu_x'] * s['mu_y'])) / (s['sigma_x'] * s['sigma_y'])

def feat_homogeneity(P):
    s = get_stats(P)
    return np.sum(P / (1 + (s['i'] - s['j'])**2))

def feat_invdiff(P):
    s = get_stats(P)
    return np.sum(P / (1 + np.abs(s['i'] - s['j'])))

def feat_diffavg(P):
    s = get_stats(P)
    diff_matrix = np.abs(s['i'] - s['j'])
    dissimilarity = np.sum(diff_matrix * s['P'])
    return dissimilarity

def feat_diffent(P):
    s = get_stats(P)
    p_diff = s['p_diff']
    p_diff_nonzero = p_diff[p_diff > 1e-15]
    diff_entropy = -np.sum(p_diff_nonzero * np.log(p_diff_nonzero))
    return diff_entropy

def feat_diffvar(P):
    s = get_stats(P)
    p_diff = s['p_diff']
    N = s['N']
    k_values = np.arange(N)
    mu_diff = np.sum(k_values * p_diff)
    diff_variance = np.sum(((k_values - mu_diff)**2) * p_diff)
    return diff_variance

def feat_imc1(P):
    s = get_stats(P)
    eps = 1e-15 
    hxy = -np.sum(s['P'] * np.log(s['P'] + eps))
    hx = -np.sum(s['p_x'] * np.log(s['p_x'] + eps))
    hy = -np.sum(s['p_y'] * np.log(s['p_y'] + eps))
    marginal_product = np.outer(s['p_x'], s['p_y'])
    hxy1 = -np.sum(s['P'] * np.log(marginal_product + eps))
    
    denominator = max(hx, hy)
    if denominator < eps:
        return 0.0
    imc1 = (hxy - hxy1) / denominator
    return imc1

def feat_imc2(P):
    s = get_stats(P)
    eps = 1e-15 
    hxy = -np.sum(s['P'] * np.log(s['P'] + eps))
    hx = -np.sum(s['p_x'] * np.log(s['p_x'] + eps))
    hy = -np.sum(s['p_y'] * np.log(s['p_y'] + eps))
    hxy2 = hx + hy
    
    term = 1.0 - np.exp(-2.0 * (hxy2 - hxy))
    imc2 = np.sqrt(max(0, term))
    return imc2

def feat_jointavg(P):
    s = get_stats(P)
    return np.sum(s['i'] * s['P'])

def feat_sumavg(P):
    s = get_stats(P)
    return np.sum((s['i'] + s['j']) * s['P'])

def feat_sument(P):
    s = get_stats(P)
    p_sum = s['p_sum']
    p_sum_nonzero = p_sum[p_sum > 1e-15]
    return -np.sum(p_sum_nonzero * np.log(p_sum_nonzero))

def feat_sumsqr(P):
    s = get_stats(P)
    return np.sum((s['i'] - s['mu_x'])**2 * s['P'])

def feat_invvar(P):
    s = get_stats(P)
    p_diff = s['p_diff']
    N = s['N']
    k_values = np.arange(1, N)
    relevant_p_diff = p_diff[1:]
    return np.sum(relevant_p_diff / (k_values**2))

## Autocorrelation
$$ \sum^{N}_{i=1} \sum^{N}_{j=1} (i \cdot j) p(i, j) $$

A GLCM that would maximize this function, would be one that is concentratwed at the diagonals of greater entries.

lets try to get as many combinations of 8 by 8 images that have the highest concentration in the bottom corner entries.

In [15]:
opt_glcm, val = maximize_haralick(feat_autocorr, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 4), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]]  
 With 16.0


In [16]:
opt_glcm, val = minimize_haralick(feat_autocorr, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 4), " \n With", np.round(val, 3))

Optimal Matrix:
 [[1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]  
 With -1.0


## Cluster Prominence
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i + j - \mu_x - \mu_y)^4 p(i, j) $$

In [17]:
opt_glcm, val = maximize_haralick(feat_cluster_prominence, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 4), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.5 0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0.5]]  
 With 81.0


In [18]:
opt_glcm, val = minimize_haralick(feat_autocorr, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 4), " \n With", np.round(val, 3))

Optimal Matrix:
 [[1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]  
 With -1.0


## Cluster Shade


$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i + j - \mu_x - \mu_y)^3 p(i, j) $$


In [19]:
opt_glcm, val = minimize_haralick(feat_cluster_shade, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.211 0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.789]]  
 With 20.785


## Cluster Tendency

$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i + j - \mu_x - \mu_y)^2 p(i, j) $$

In [20]:
opt_glcm, val = maximize_haralick(feat_cluster_tendency, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 4), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.5 0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0.5]]  
 With 9.0


In [21]:
opt_glcm, val = minimize_haralick(feat_cluster_tendency, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.   0.   0.   0.25]
 [0.   0.   0.25 0.  ]
 [0.   0.25 0.   0.  ]
 [0.25 0.   0.   0.  ]]  
 With -0.0


## Contrast
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} (i - j)^2 p(i, j) $$

In [22]:
opt_glcm, val = maximize_haralick(feat_contrast, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.  0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0. ]]  
 With 9.0


In [23]:
opt_glcm, val = minimize_haralick(feat_contrast, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.  ]
 [0.   0.   0.25 0.  ]
 [0.   0.   0.   0.25]]  
 With -0.0


## Correlation
$$ \frac{\sum^{N_g}_{i=1}\sum^{N_g}_{j=1}{p(i,j)ij-\mu_x\mu_y}}{\sigma_x(i)\sigma_y(j)} $$

In [24]:
opt_glcm, val = maximize_haralick(feat_correlation, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.5 0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0.5]]  
 With 1.0


In [25]:
opt_glcm, val = minimize_haralick(feat_correlation, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.  0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0. ]]  
 With 1.0


## Difference Average / Dissimilarity

$$ \sum^{N}_{i=1}\sum^{N}_{j=1}{|i-j|p(i,j)} $$

In [26]:
opt_glcm, val = maximize_haralick(feat_diffavg, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.  0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0. ]]  
 With 3.0


In [27]:
opt_glcm, val = minimize_haralick(feat_diffavg, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.  ]
 [0.   0.   0.25 0.  ]
 [0.   0.   0.   0.25]]  
 With -0.0


## Difference entropy

$$ -\sum_{k=0}^{N-1} p_{x-y}(k) \log p_{x-y}(k) $$

In [28]:
opt_glcm, val = maximize_haralick(feat_diffent, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.063 0.042 0.063 0.125]
 [0.042 0.063 0.042 0.063]
 [0.063 0.042 0.063 0.042]
 [0.125 0.063 0.042 0.063]]  
 With 1.386


In [29]:
opt_glcm, val = minimize_haralick(feat_diffent, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.    0.167 0.    0.   ]
 [0.167 0.    0.167 0.   ]
 [0.    0.167 0.    0.167]
 [0.    0.    0.167 0.   ]]  
 With -0.0


## Difference variance

$$ \sum_{k=0}^{N-1} (k - \mu_{x-y})^2 p_{x-y}(k) $$


In [30]:
opt_glcm, val = maximize_haralick(feat_diffvar, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.125 0.    0.    0.25 ]
 [0.    0.125 0.    0.   ]
 [0.    0.    0.125 0.   ]
 [0.25  0.    0.    0.125]]  
 With 2.25


In [31]:
opt_glcm, val = minimize_haralick(feat_diffvar, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.    0.167 0.    0.   ]
 [0.167 0.    0.167 0.   ]
 [0.    0.167 0.    0.167]
 [0.    0.    0.167 0.   ]]  
 With -0.0


## Energy
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} p(i, j)^2 $$

In [32]:
opt_glcm, val = maximize_haralick(feat_asm, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With 0.062


In [33]:
opt_glcm, val = minimize_haralick(feat_asm, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With -0.062


## Entropy
$$ -\sum_{i=1}^{N} \sum_{j=1}^{N} p(i, j) \log p(i, j) $$

In [34]:
opt_glcm, val = maximize_haralick(feat_entropy, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With 2.773


In [35]:
opt_glcm, val = minimize_haralick(feat_entropy, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With -2.773


## Homogeneity (2)
$$ \sum_{i=1}^{N} \sum_{j=1}^{N} \frac{p(i, j)}{1 + (i - j)^2} $$

In [36]:
opt_glcm, val = maximize_haralick(feat_homogeneity, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.  ]
 [0.   0.   0.25 0.  ]
 [0.   0.   0.   0.25]]  
 With 1.0


In [37]:
opt_glcm, val = minimize_haralick(feat_homogeneity, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.  0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0. ]]  
 With -0.1


## Information measure of correlation 1

$$ \frac{HXY-HXY1}{\max\{HX,HY\}} $$

In [38]:
opt_glcm, val = maximize_haralick(feat_imc1, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With 0.0


In [39]:
opt_glcm, val = minimize_haralick(feat_imc1, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With -0.0


## Information measure of correlation 2

$$ \sqrt{1-e^{-2(HXY2-HXY)}} $$

In [40]:
opt_glcm, val = maximize_haralick(feat_imc2, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With 0.0


In [41]:
opt_glcm, val = minimize_haralick(feat_imc2, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With -0.0


## Inverse difference

$$ \sum_{i=1}^{N} \sum_{j=1}^{N} \frac{p(i, j)}{1 + |i - j|} $$

In [42]:
opt_glcm, val = maximize_haralick(feat_invdiff, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.  ]
 [0.   0.   0.25 0.  ]
 [0.   0.   0.   0.25]]  
 With 1.0


In [43]:
opt_glcm, val = minimize_haralick(feat_invdiff, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.  0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0. ]]  
 With -0.25


## Joint average

$$ \sum_{i=1}^{N} \sum_{j=1}^{N} i \cdot p(i, j) $$

In [44]:
opt_glcm, val = maximize_haralick(feat_jointavg, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.   0.   0.   0.  ]
 [0.   0.   0.   0.  ]
 [0.   0.   0.   0.  ]
 [0.25 0.25 0.25 0.25]]  
 With 4.0


In [45]:
opt_glcm, val = minimize_haralick(feat_jointavg, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.25 0.25 0.25 0.25]
 [0.   0.   0.   0.  ]
 [0.   0.   0.   0.  ]
 [0.   0.   0.   0.  ]]  
 With -1.0


## Sum average

$$ \sum^{2N}_{k=2}{k \cdot p_{x+y}(k)} $$

In [46]:
opt_glcm, val = maximize_haralick(feat_sumavg, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]]  
 With 8.0


In [47]:
opt_glcm, val = minimize_haralick(feat_sumavg, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]  
 With -2.0


## Sum entropy

$$  - \sum^{2N}_{k=2}{p_{x+y}(k)\log p_{x+y}(k)} $$

In [48]:
opt_glcm, val = maximize_haralick(feat_sument, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.143 0.071 0.048 0.036]
 [0.071 0.048 0.036 0.048]
 [0.048 0.036 0.048 0.071]
 [0.036 0.048 0.071 0.143]]  
 With 1.946


In [49]:
opt_glcm, val = minimize_haralick(feat_sument, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.   0.   0.   0.25]
 [0.   0.   0.25 0.  ]
 [0.   0.25 0.   0.  ]
 [0.25 0.   0.   0.  ]]  
 With -0.0


## Sum of squares

$$ \sum^{N}_{i=1}\sum^{N}_{j=1}{(i-\mu_x)^2p(i,j)} $$

In [50]:
opt_glcm, val = maximize_haralick(feat_sumsqr, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.125 0.125 0.125 0.125]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.125 0.125 0.125 0.125]]  
 With 2.25


In [51]:
opt_glcm, val = minimize_haralick(feat_sumsqr, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.    0.    0.    0.   ]
 [0.125 0.125 0.125 0.125]
 [0.125 0.125 0.125 0.125]
 [0.    0.    0.    0.   ]]  
 With -0.25


## Maximal Correlation Coefficient

$$ \sqrt{\lambda_2 Q(i,j)} $$

$$ Q(i, j) = \displaystyle\sum^{N}_{k=0}{\frac{p(i,k)p(j, k)}{p_x(i)p_y(k)}} $$

In [52]:
def feat_mcc(P):
    s = get_stats(P)
    N = s['N']
    P = s['P']
    px = s['p_x']
    py = s['p_y']
    eps = 1e-15

    Q = np.zeros((N, N))

    px_inv = 1.0 / (px + eps)
    py_inv = 1.0 / (py + eps)
    
    for i in range(N):
        for j in range(N):
            term = (P[i, :] * P[j, :]) * px_inv[i] * py_inv
            Q[i, j] = np.sum(term)

    evals = np.linalg.eigvals(Q)
    
    evals = np.sort(np.real(evals))
    
    if len(evals) >= 2:
        mcc = np.sqrt(max(0, evals[-2]))
    else:
        mcc = 0.0
        
    return mcc

In [53]:
opt_glcm, val = maximize_haralick(feat_mcc, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.063 0.063 0.063 0.063]
 [0.062 0.062 0.062 0.062]
 [0.063 0.063 0.063 0.063]
 [0.063 0.063 0.063 0.063]]  
 With 0.0


In [54]:
opt_glcm, val = minimize_haralick(feat_mcc, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.062 0.062 0.062 0.062]
 [0.063 0.063 0.063 0.063]
 [0.062 0.062 0.062 0.062]
 [0.062 0.062 0.062 0.062]]  
 With -0.0


## Inverse variance

$$ \sum^{N-1}_{k=1}{\frac{p_{x-y}(k)}{k^2}} $$

In [55]:
opt_glcm, val = maximize_haralick(feat_invvar, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.    0.167 0.    0.   ]
 [0.167 0.    0.167 0.   ]
 [0.    0.167 0.    0.167]
 [0.    0.    0.167 0.   ]]  
 With 1.0


In [56]:
opt_glcm, val = minimize_haralick(feat_invvar, n_levels=4)
print("Optimal Matrix:\n", np.round(opt_glcm, 3), " \n With", np.round(val, 3))

Optimal Matrix:
 [[0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.  ]
 [0.   0.   0.25 0.  ]
 [0.   0.   0.   0.25]]  
 With -0.0
